# RQ5: How does hyperparameter tuning improve the best fraud detection model?

**Research Question:** To what extent does hyperparameter optimization (GridSearchCV / RandomizedSearchCV) improve the performance of the best-performing model identified in RQ4?

**Hypothesis:** Tuned XGBoost will show statistically significant improvement over default parameters, particularly in F1 and AUC-ROC.

**Target Variable:** `is_fraud`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix,
                              ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print('Libraries loaded.')

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
DATA_PATH = '/kaggle/input/financial-transactions-dataset-for-fraud-detection/financial_fraud_detection_dataset.csv'
df = pd.read_csv(DATA_PATH, nrows=50000)

TARGET = 'is_fraud'
DROP_COLS = ['transaction_id','timestamp','sender_account','receiver_account',
             'fraud_type','ip_address','device_hash']
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

df.dropna(inplace=True)
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Split done.')

In [ ]:
# ── Default XGBoost ───────────────────────────────────────────────────────────
xgb_default = XGBClassifier(n_estimators=100, random_state=42,
                             eval_metric='logloss', verbosity=0)
xgb_default.fit(X_train, y_train)
preds_def = xgb_default.predict(X_test)
proba_def = xgb_default.predict_proba(X_test)[:,1]
print('Default XGBoost done.')

In [ ]:
# ── RandomizedSearchCV Tuning ─────────────────────────────────────────────────
param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [3, 5, 7, 9],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'subsample':         [0.6, 0.8, 1.0],
    'colsample_bytree':  [0.6, 0.8, 1.0],
    'min_child_weight':  [1, 3, 5],
    'gamma':             [0, 0.1, 0.3],
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
search = RandomizedSearchCV(
    xgb_base, param_dist,
    n_iter=20, scoring='roc_auc',
    cv=3, random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV AUC:', round(search.best_score_, 4))

In [ ]:
xgb_tuned = search.best_estimator_
preds_tuned = xgb_tuned.predict(X_test)
proba_tuned = xgb_tuned.predict_proba(X_test)[:,1]

# ── Cross-validation scores ────────────────────────────────────────────────────
cv_default = cross_val_score(xgb_default, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
cv_tuned   = cross_val_score(xgb_tuned,   X, y, cv=5, scoring='roc_auc', n_jobs=-1)

def metrics_dict(name, preds, proba, cv_scores):
    return {
        'Model':       name,
        'Accuracy':    round(accuracy_score(y_test, preds), 4),
        'Precision':   round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall':      round(recall_score(y_test, preds, zero_division=0), 4),
        'F1':          round(f1_score(y_test, preds, zero_division=0), 4),
        'AUC-ROC':     round(roc_auc_score(y_test, proba), 4),
        'CV_AUC_Mean': round(cv_scores.mean(), 4),
        'CV_AUC_Std':  round(cv_scores.std(), 4),
    }

df_res = pd.DataFrame([
    metrics_dict('XGBoost Default', preds_def, proba_def, cv_default),
    metrics_dict('XGBoost Tuned',   preds_tuned, proba_tuned, cv_tuned),
])
df_res

In [ ]:
# Save best params
df_params = pd.DataFrame([search.best_params_])
df_params.to_csv('table_rq5_best_params.csv', index=False)
df_res.to_csv('table_rq5_tuning_comparison.csv', index=False)
print('Tables saved.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Left: Metric comparison bar
metrics = ['Accuracy','Precision','Recall','F1','AUC-ROC']
x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w/2, df_res[df_res['Model']=='XGBoost Default'][metrics].values[0],
            w, label='Default', color='#2196F3', alpha=0.85)
axes[0].bar(x + w/2, df_res[df_res['Model']=='XGBoost Tuned'][metrics].values[0],
            w, label='Tuned', color='#4CAF50', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Default vs Tuned\nXGBoost Metrics', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[0].set_axisbelow(True)

# Middle: ROC curves
fpr_d, tpr_d, _ = roc_curve(y_test, proba_def)
fpr_t, tpr_t, _ = roc_curve(y_test, proba_tuned)
axes[1].plot(fpr_d, tpr_d, color='#2196F3', lw=2,
             label=f'Default (AUC={roc_auc_score(y_test,proba_def):.3f})')
axes[1].plot(fpr_t, tpr_t, color='#4CAF50', lw=2,
             label=f'Tuned (AUC={roc_auc_score(y_test,proba_tuned):.3f})')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curves\nDefault vs Tuned', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.6)

# Right: Cross-validation box
axes[2].boxplot([cv_default, cv_tuned], labels=['Default','Tuned'],
                patch_artist=True,
                boxprops=dict(facecolor='#E3F2FD'),
                medianprops=dict(color='#1565C0', linewidth=2))
axes[2].set_ylabel('CV AUC-ROC (5-fold)', fontsize=11)
axes[2].set_title('Cross-Validation\nAUC Distribution', fontsize=11, fontweight='bold')
axes[2].yaxis.grid(True, linestyle='--', alpha=0.6)

plt.suptitle('RQ5: Hyperparameter Tuning Impact on XGBoost Fraud Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figure_rq5_hyperparameter_tuning.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved.')